In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
%cd /content/drive/MyDrive/plant_health_3class_project
!ls

/content/drive/MyDrive/plant_health_3class_project
app	configs  deliverables  outputs		README.md	  src
app.py	data	 notebooks     PROJECT_PLAN.md	requirements.txt


In [3]:
   !pip install torch torchvision pycocotools pandas matplotlib pillow tqdm pyyaml

In [8]:
!python src/split_coco_dataset.py \
--images data/images/all \
--annotations data/annotations/annotations_coco.json \
--out data

train
Images: 155
Annotations: 724
Missing: 0
Saved: data/annotations/train.json
val
Images: 19
Annotations: 75
Missing: 0
Saved: data/annotations/val.json
test
Images: 20
Annotations: 83
Missing: 0
Saved: data/annotations/test.json


In [5]:
!python src/validate_coco.py --annotations data/annotations/train.json --image_dir data/images/train

!python src/validate_coco.py --annotations data/annotations/val.json --image_dir data/images/val

!python src/validate_coco.py --annotations data/annotations/test.json --image_dir data/images/test

Images: 155
Annotations: 724
Categories: {1: 'healthy_leaf', 2: 'healthy_stem', 3: 'diseased_leaf'}
Counts:
  1 healthy_leaf: 274
  2 healthy_stem: 199
  3 diseased_leaf: 251
Missing image files: 0
Images: 19
Annotations: 75
Categories: {1: 'healthy_leaf', 2: 'healthy_stem', 3: 'diseased_leaf'}
Counts:
  1 healthy_leaf: 26
  2 healthy_stem: 18
  3 diseased_leaf: 31
Missing image files: 0
Images: 20
Annotations: 83
Categories: {1: 'healthy_leaf', 2: 'healthy_stem', 3: 'diseased_leaf'}
Counts:
  1 healthy_leaf: 38
  2 healthy_stem: 20
  3 diseased_leaf: 25
Missing image files: 0


In [6]:
!python src/evaluate_baseline_no_training.py \
--config configs/maskrcnn_config.yaml \
--split val \
--threshold 0.5

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth
100% 170M/170M [00:01<00:00, 168MB/s]
Evaluating: 100% 19/19 [00:41<00:00,  2.16s/it]
Evaluation Results
score_threshold: 0.5000
mean_iou: 0.0006
mean_dice: 0.0011
avg_inference_time: 0.2590


In [8]:
!python src/train_maskrcnn.py --config configs/maskrcnn_config.yaml

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Training: 100% 78/78 [03:18<00:00,  2.55s/it]
Validation: 100% 10/10 [00:23<00:00,  2.33s/it]
Epoch 1: phase=head_only, train_loss=1.7565, val_loss=1.3021
Saved best model: outputs/checkpoints/maskrcnn_best.pth
Training: 100% 78/78 [02:29<00:00,  1.92s/it]
Validation: 100% 10/10 [00:14<00:00,  1.49s/it]
Epoch 2: phase=head_only, train_loss=1.2687, val_loss=1.2325
Saved best model: outputs/checkpoints/maskrcnn_best.pth
Training: 100% 78/78 [02:28<00:00,  1.90s/it]
Validation: 100% 10/10 [00:14<00:00,  1.48s/it]
Epoch 3: phase=head_only, train_loss=1.1792, val_loss=1.1945
Saved best model: outputs/checkpoints/maskrcnn_best.pth
Unfreezing backbone for full fine-tuning...
Training: 100% 78/78 [02:49<00:00,  2.17s/it]
Validation: 100% 10/10 [00:14<00:00,  1.49s/it]
Epoch 4: phase=full_finetune, train_loss=1.1329, val_loss=1.169

In [3]:
!python src/random_search_maskrcnn.py \
--config configs/maskrcnn_config.yaml \
--max_trials 4 \
--epochs 5


=== Trial 1: lr=0.001, batch=1, wd=0.0001 ===
loading annotations into memory...
Done (t=0.76s)
creating index...
index created!
loading annotations into memory...
Done (t=0.37s)
creating index...
index created!
Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth
100% 170M/170M [00:01<00:00, 142MB/s]
Training: 100% 155/155 [03:24<00:00,  1.32s/it]
Validation: 100% 19/19 [00:21<00:00,  1.12s/it]
Epoch 1: phase=head_only, train_loss=1.4332, val_loss=1.1778
Saved best model: outputs/hparam/trial_1/maskrcnn_best.pth
Training: 100% 155/155 [02:26<00:00,  1.06it/s]
Validation: 100% 19/19 [00:14<00:00,  1.31it/s]
Epoch 2: phase=head_only, train_loss=1.1274, val_loss=1.1306
Saved best model: outputs/hparam/trial_1/maskrcnn_best.pth
Training: 100% 155/155 [02:27<00:00,  1.05it/s]
Validation: 100% 19/19 [00:14<00:00,  1.28it/s]
Epoch 3: phase=head_only, train_loss=1.0677, val_los

In [4]:
!python src/evaluate_maskrcnn.py \
--config configs/maskrcnn_config.yaml \
--weights outputs/checkpoints/maskrcnn_best.pth \
--split test \
--threshold 0.5

loading annotations into memory...
Done (t=0.49s)
creating index...
index created!
Evaluating: 100% 20/20 [00:31<00:00,  1.59s/it]
Evaluation Results
score_threshold: 0.5000
mean_iou: 0.3385
mean_dice: 0.3990
avg_inference_time: 0.1790


In [5]:
!python src/predict_maskrcnn.py \
--image data/images/test/IMG_6932.jpg \
--weights outputs/checkpoints/maskrcnn_best.pth \
--threshold 0.5 \
--top_k 6 \
--tta

Raw predictions: 148
After threshold: 13
After class-wise NMS: 6
Final predictions shown: 6
Saved prediction to: outputs/predictions/prediction.png


In [6]:
!python src/full_visualizations.py

Saved all visualizations to outputs/visualizations/


In [9]:
!python src/coco_to_semantic_masks.py --annotations data/annotations/train.json --output_dir data/semantic_masks/train
!python src/coco_to_semantic_masks.py --annotations data/annotations/val.json --output_dir data/semantic_masks/val

loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
Saved semantic masks to: data/semantic_masks/train
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Saved semantic masks to: data/semantic_masks/val


In [11]:
 !rm -rf data/semantic_masks/train
!rm -rf data/semantic_masks/val

!python src/coco_to_semantic_masks.py --annotations data/annotations/train.json --output_dir data/semantic_masks/train
!python src/coco_to_semantic_masks.py --annotations data/annotations/val.json --output_dir data/semantic_masks/val

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Saved semantic masks to: data/semantic_masks/train
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Saved semantic masks to: data/semantic_masks/val


In [12]:
!ls data/semantic_masks/train/IMG_6939.png
!ls data/images/train/IMG_6939.*

ls: cannot access 'data/semantic_masks/train/IMG_6939.png': No such file or directory
data/images/train/IMG_6939.jpg


In [13]:
import os
from PIL import Image
import numpy as np

image_dir = "data/images/train"
mask_dir = "data/semantic_masks/train"

os.makedirs(mask_dir, exist_ok=True)

missing = []

for file in os.listdir(image_dir):
    if file.lower().endswith((".jpg", ".jpeg", ".png")):
        base = os.path.splitext(file)[0]
        mask_path = os.path.join(mask_dir, base + ".png")

        if not os.path.exists(mask_path):
            img = Image.open(os.path.join(image_dir, file))
            blank_mask = np.zeros((512, 512), dtype=np.uint8)
            Image.fromarray(blank_mask).save(mask_path)
            missing.append(file)

print("Created blank masks for:", len(missing))
print(missing[:10])

Created blank masks for: 33
['IMG_6919.jpg', 'IMG_6915.jpg', 'IMG_6939.jpg', '20260330_192802.JPG', 'WhatsApp Image 2026-04-28 at 23.14.41 (3).jpeg', 'WhatsApp Image 2026-04-28 at 23.14.41 (6).jpeg', 'WhatsApp Image 2026-04-28 at 23.14.42 (1).jpeg', 'IMG_8424.jpeg', 'IMG_8437.jpeg', 'IMG_8443.jpeg']


In [16]:
import os
from PIL import Image
import numpy as np

image_dir = "data/images/val"
mask_dir = "data/semantic_masks/val"

os.makedirs(mask_dir, exist_ok=True)

missing = []

for file in os.listdir(image_dir):
    if file.lower().endswith((".jpg", ".jpeg", ".png")):
        base = os.path.splitext(file)[0]
        mask_path = os.path.join(mask_dir, base + ".png")

        if not os.path.exists(mask_path):
            blank_mask = np.zeros((512, 512), dtype=np.uint8)
            Image.fromarray(blank_mask).save(mask_path)
            missing.append(file)

print("Created blank validation masks for:", len(missing))
print(missing[:10])

Created blank validation masks for: 14
['WhatsApp Image 2026-04-28 at 23.14.41.jpeg', 'WhatsApp Image 2026-04-28 at 23.14.41 (4).jpeg', 'WhatsApp Image 2026-04-28 at 23.14.41 (5).jpeg', 'WhatsApp Image 2026-04-28 at 23.14.41 (8).jpeg', 'IMG_8426.jpeg', 'IMG_8435.jpeg', 'IMG_8430.jpeg', 'IMG_8448.jpeg', 'IMG_7632.jpg', 'IMG_7594.jpg']


In [17]:
!python src/train_unet.py --config configs/unet_config.yaml

Train: 100% 45/45 [01:00<00:00,  1.33s/it]
Val: 100% 9/9 [00:12<00:00,  1.44s/it]
Epoch 1: train_loss=0.9923, train_acc=0.6687, val_loss=0.7978, val_acc=0.7968
Saved best model: outputs/checkpoints/mini_unet_best.pth
Train: 100% 45/45 [00:59<00:00,  1.32s/it]
Val: 100% 9/9 [00:09<00:00,  1.07s/it]
Epoch 2: train_loss=0.7911, train_acc=0.7444, val_loss=0.6596, val_acc=0.8316
Saved best model: outputs/checkpoints/mini_unet_best.pth
Train: 100% 45/45 [01:00<00:00,  1.34s/it]
Val: 100% 9/9 [00:09<00:00,  1.06s/it]
Epoch 3: train_loss=0.7581, train_acc=0.7443, val_loss=0.6244, val_acc=0.8211
Saved best model: outputs/checkpoints/mini_unet_best.pth
Train: 100% 45/45 [01:00<00:00,  1.34s/it]
Val: 100% 9/9 [00:09<00:00,  1.05s/it]
Epoch 4: train_loss=0.7456, train_acc=0.7460, val_loss=0.6194, val_acc=0.8288
Saved best model: outputs/checkpoints/mini_unet_best.pth
Train: 100% 45/45 [00:59<00:00,  1.32s/it]
Val: 100% 9/9 [00:09<00:00,  1.09s/it]
Epoch 5: train_loss=0.7307, train_acc=0.7442, val_

In [6]:
!touch app.py

In [7]:
pip install gradio

In [ ]:
!python app.py

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://8c5d7d4bf5bfb9a6ca.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [10]:
!sed -n '1,200p' src/models/maskrcnn_model.py

import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

def get_maskrcnn_model(num_classes):
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights="DEFAULT")

    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    model.roi_heads.mask_predictor = MaskRCNNPredictor(
        in_features_mask,
        hidden_layer,
        num_classes
    )

    return model
